# Part 4: Make your own agent

An agent is an LLM, some tools, and a loop. This one gets three tools, two for the network and one for the cluster, in about 40 lines of Python: no framework, just the `openai` package and NRP's endpoint.

**Kernel:** Python 3. On the training hub, `OPENAI_API_KEY` already holds an LLM token; elsewhere, get one at [nrp.ai/llmtoken](https://nrp.ai/llmtoken) and set it first.

> 📘 [LLM API access](https://nrp.ai/documentation/userdocs/ai/llm-managed/api-access/) · [Models and what they're good at](https://nrp.ai/documentation/userdocs/ai/llm-managed/models/)

In [ ]:
%pip install -q openai

In [ ]:
import json, os, socket, subprocess, time
from openai import OpenAI

client = OpenAI(base_url="https://ellm.nrp-nautilus.io/v1", api_key=os.environ["OPENAI_API_KEY"])
MODEL = "gpt-oss"    # reliable at calling tools; the task name "agents" works too

## How an agent works

```
   ┌────────────┐  asks for a tool?  ┌──────────────────────┐
   │  call LLM  │ ──────── yes ────► │ your code runs it,   │
   └────────────┘                    │ adds the result      │
         │   ▲                       └──────────┬───────────┘
         no  └──────────────────────────────────┘
         │
    the answer
```

The model never runs anything. It replies "please call `dns_lookup` with `name=...`"; **your** code runs the function and sends the result back. The growing list of messages is the agent's whole state.

## 1. The tools: plain Python functions

In [ ]:
def dns_lookup(name):
    """The addresses a host name resolves to."""
    try:
        return json.dumps(sorted({a[4][0] for a in socket.getaddrinfo(name, 443)}))
    except socket.gaierror as e:
        return f"ERROR: {name} doesn't resolve: {e}"


def tcp_connect(host, port):
    """Open a TCP connection; say whether it worked and how long it took."""
    start = time.time()
    try:
        socket.create_connection((host, int(port)), timeout=3).close()
        return f"{host}:{port} open, connected in {(time.time() - start) * 1000:.1f} ms"
    except OSError as e:
        return f"{host}:{port} not reachable: {e}"


def job_status(job):
    """A Job's pod in my namespace: its phase, its node, and the last lines of its log."""
    pods = subprocess.run(["kubectl", "get", "pods", "-l", f"job-name={job}", "-o",
                           "custom-columns=POD:.metadata.name,PHASE:.status.phase,NODE:.spec.nodeName"],
                          capture_output=True, text=True)
    logs = subprocess.run(["kubectl", "logs", f"job/{job}", "--tail=5"], capture_output=True, text=True)
    return pods.stdout + pods.stderr + "last log lines:\n" + (logs.stdout or logs.stderr)


print(dns_lookup("ellm.nrp-nautilus.io"))
print(tcp_connect("ellm.nrp-nautilus.io", 443))

## 2. Describe them to the model

Each tool gets a schema: its name, what it does, and its arguments. The descriptions are prompt text: the model decides what to call from them alone.

In [ ]:
def tool(fn, description, **params):
    """The JSON schema that tells the model a tool exists."""
    return {"type": "function", "function": {
        "name": fn.__name__, "description": description,
        "parameters": {"type": "object", "properties": {k: {"type": v} for k, v in params.items()},
                       "required": list(params)}}}


TOOLS = {f.__name__: f for f in (dns_lookup, tcp_connect, job_status)}
SCHEMAS = [
    tool(dns_lookup, "Resolve a host name to its IPv4 and IPv6 addresses.", name="string"),
    tool(tcp_connect, "Try a TCP connection to host:port; says whether it opened, and the connect time in ms.",
         host="string", port="integer"),
    tool(job_status, "Status of a Kubernetes Job in my namespace: pod phase, node, and its last log lines.",
         job="string"),
]

## 3. The loop

In [ ]:
SYSTEM = "You help a network engineer. Check facts with the tools instead of guessing, then answer briefly."


def run_agent(question, model=MODEL, max_turns=8):
    messages = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": question}]
    for turn in range(max_turns):
        msg = client.chat.completions.create(model=model, messages=messages, tools=SCHEMAS).choices[0].message
        if not msg.tool_calls:                             # no tool requested: that's the answer
            return msg.content
        messages.append({"role": "assistant", "content": msg.content or "",
                         "tool_calls": [c.model_dump() for c in msg.tool_calls]})   # its request...
        for call in msg.tool_calls:
            args = json.loads(call.function.arguments)
            try:
                result = TOOLS[call.function.name](**args)  # ...your code runs the tool...
            except Exception as e:
                result = f"ERROR: {e}"                     # errors go back as text the model can read
            print(f"[turn {turn}] {call.function.name}({args}) -> {result.strip()[:120]}")
            messages.append({"role": "tool", "tool_call_id": call.id, "content": result})   # ...and reports
    return "Stopped: too many turns."

## 4. Ask it something

In [ ]:
print(run_agent("Do gitlab.nrp-nautilus.io, ellm.nrp-nautilus.io and perfsonar.nrp-nautilus.io resolve to the "
                "same address? Is port 443 open on each, and how fast does it connect?"))

Watch the `[turn N]` lines: it looked up each name, then tested each one, and nobody wrote that plan. All three names land on the same address: one of the ingress nodes from the *Getting Traffic In* slide. Run it again and the order may change; that's normal for agents.

Now your GPU job from part 3. **Edit this cell**: put the same short name you used there in `ME`.

In [ ]:
ME = "changeme"
print(run_agent(f"Is my job gpu-train-{ME} finished? Which node did it run on, and what did it print last?"))

## Why this is safe to run

- The model executes nothing: only the three functions in `TOOLS` can run, with arguments your code parses.
- The tools only read, and `kubectl` runs as your hub account, in your namespace.
- `max_turns` stops a model that loops; errors go back as text, so it can correct itself.

That's the same design as **nrp-ops-bot** from the talk, with 16 tools instead of 3.

**Try:** add a fourth tool, for example `http_status(url)` with `urllib.request`. Register it in `TOOLS` and `SCHEMAS`, then ask which of the `*.nrp-nautilus.io` services answer with 200.